In [ ]:
import os
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification
)
import evaluate

# 1. Setup & Metadata
MODEL_NAMES = ["lukasweber/WG_BERT", "roberta-base"]
labels = ["O", "B-VehicleMakeModel", "I-VehicleMakeModel", "B-FaultCode", "I-FaultCode", 
          "B-CustomerReportedSymptom", "I-CustomerReportedSymptom", "B-VehicleComponentLocation", 
          "I-VehicleComponentLocation", "B-MaintenanceMethod", "I-MaintenanceMethod", 
          "B-DeviceProperties", "I-DeviceProperties", "B-MeasurementValue", "I-MeasurementValue", 
          "B-EquipmentName", "I-EquipmentName", "B-FaultType", "I-FaultType", "B-TimeDuration", "I-TimeDuration"]
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

# 2. Metric Computation
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [[id2label[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
    true_labels = [[id2label[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 3. Training Loop (Generalized for both models)
def train_model(model_name):
    print(f"--- Training {model_name} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(labels), id2label=id2label, label2id=label2id)
    
    # [Insert tokenization logic from your notebooks here]
    
    args = TrainingArguments(
        output_dir=f"./output-{model_name.split('/')[-1]}",
        evaluation_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        save_total_limit=1
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorForTokenClassification(tokenizer)
    )
    
    trainer.train()
    return trainer

# Execute training for both
# wg_bert_trainer = train_model("lukasweber/WG_BERT")
# roberta_trainer = train_model("roberta-base")